In [1]:
import pandas as pd
import io
import numpy as np

In [2]:
# ==============================================================================
# Step 1: Based on the data in worksheet "raw" to create a dataframe called "df_target"
# ==============================================================================
# In a real-world scenario, you would use pd.read_excel('your_file.xlsx', sheet_name='raw')
# For this example, we'll create the DataFrame from the provided text data.
raw_data = """cust_id,rule_10,rule_12,rule_15,rule_18,rule_20
1,1,0,0,0,1
2,1,0,0,1,1
3,1,0,0,0,1
4,0,0,0,1,0
5,0,0,0,0,0
6,0,0,1,1,0
7,1,0,1,0,0
8,1,0,0,1,1
9,0,0,0,0,0
10,0,0,0,1,1
"""
df_target = pd.read_csv(io.StringIO(raw_data))

In [3]:
# ==============================================================================
# Step 2: Create the key DataFrame
# ==============================================================================
# Again, we create the DataFrame from the provided text data.
# Information from both 'key' and 'result' worksheets is combined to create a complete key.
key_data = """rule,exclusion_reason
rule_10,Invalid ID
rule_12,No address
rule_15,Foreign
rule_18,No credit card
rule_20,Wrong promo code
"""
df_key_source = pd.read_csv(io.StringIO(key_data))

In [4]:
# Get all 'rule_*' column names from df_target and sort them.
rule_columns = sorted([col for col in df_target.columns if col.startswith('rule_')])

# Create the final df_key by merging the full rule list with the source key data.
df_key = pd.DataFrame({'rule': rule_columns})
df_key = pd.merge(df_key, df_key_source, on='rule', how='left')

In [5]:
print(df_key)

      rule  exclusion_reason
0  rule_10        Invalid ID
1  rule_12        No address
2  rule_15           Foreign
3  rule_18    No credit card
4  rule_20  Wrong promo code


In [6]:
# ==============================================================================
# Step 3: Create df_stage with the 'waterfall_hit' column
# ==============================================================================
# Create df_stage as a copy of df_target.
df_stage = df_target.copy()

# Set a default value for 'waterfall_hit'.
df_stage['waterfall_hit'] = 'rule_999'

# Iterate through the rules in reverse order to apply the waterfall logic.
# The last rule that is '1' in the sorted list will be the one that is kept.
for rule in reversed(rule_columns):
    df_stage.loc[df_stage[rule] == 1, 'waterfall_hit'] = rule
    
print(df_stage)

   cust_id  rule_10  rule_12  rule_15  rule_18  rule_20 waterfall_hit
0        1        1        0        0        0        1       rule_10
1        2        1        0        0        1        1       rule_10
2        3        1        0        0        0        1       rule_10
3        4        0        0        0        1        0       rule_18
4        5        0        0        0        0        0      rule_999
5        6        0        0        1        1        0       rule_15
6        7        1        0        1        0        0       rule_10
7        8        1        0        0        1        1       rule_10
8        9        0        0        0        0        0      rule_999
9       10        0        0        0        1        1       rule_18


In [7]:
# ==============================================================================
# Step 4: Use df_stage and df_key to generate a summary table
# ==============================================================================
# Calculate the number of customers for each 'waterfall_hit' value.
individual_hit_counts = df_stage['waterfall_hit'].value_counts()

# Create the summary table by mapping the counts to df_key.
summary_table = df_key.copy()
summary_table['individual_hit'] = summary_table['rule'].map(individual_hit_counts).fillna(0).astype(int)

# Get the total number of customers.
total_customers = len(df_stage)

# Calculate the waterfall column as the cumulative remaining customers.
summary_table['waterfall'] = total_customers - summary_table['individual_hit'].cumsum()

print(summary_table)

      rule  exclusion_reason  individual_hit  waterfall
0  rule_10        Invalid ID               5          5
1  rule_12        No address               0          5
2  rule_15           Foreign               1          4
3  rule_18    No credit card               2          2
4  rule_20  Wrong promo code               0          2


In [8]:
# ==============================================================================
# Step 5: Insert a row at the beginning of the summary table
# ==============================================================================
# Create the initial row as a dictionary.
initial_row = {
    'rule': '-',
    'exclusion_reason': '(Whole Base before exclusion)',
    'individual_hit': 0,
    'waterfall': total_customers
}

# Convert the dictionary to a DataFrame and concatenate it with the summary table.
summary_table = pd.concat([pd.DataFrame([initial_row]), summary_table], ignore_index=True)

print(summary_table)

      rule               exclusion_reason  individual_hit  waterfall
0        -  (Whole Base before exclusion)               0         10
1  rule_10                     Invalid ID               5          5
2  rule_12                     No address               0          5
3  rule_15                        Foreign               1          4
4  rule_18                 No credit card               2          2
5  rule_20               Wrong promo code               0          2


In [9]:
# ==============================================================================
# Step 6: Display the formatted summary table
# ==============================================================================
# Create a copy for display purposes to apply string formatting.
display_summary = summary_table.copy()

# Apply thousand comma formatting to numeric columns.
display_summary['individual_hit'] = display_summary['individual_hit'].apply(lambda x: f"{x:,}")
display_summary['waterfall'] = display_summary['waterfall'].apply(lambda x: f"{x:,}")

print("--- Exclusion Waterfall Summary ---")
print(display_summary.to_string(index=False))
print("\n" + "="*50 + "\n")

--- Exclusion Waterfall Summary ---
   rule              exclusion_reason individual_hit waterfall
      - (Whole Base before exclusion)              0        10
rule_10                    Invalid ID              5         5
rule_12                    No address              0         5
rule_15                       Foreign              1         4
rule_18                No credit card              2         2
rule_20              Wrong promo code              0         2




In [10]:
# ==============================================================================
# Step 7: Display the number of eligible customers
# ==============================================================================
# Get the last waterfall value, which represents the final eligible customers.
eligible_customers_count = summary_table['waterfall'].iloc[-1]
print(f"No. of eligible customers = {eligible_customers_count:,}.") # [[1]]
print("\n" + "="*50 + "\n")

No. of eligible customers = 2.




In [13]:
# ==============================================================================
# Step 8: Create df_final with only 'cust_id' and 'waterfall_hit'
# ==============================================================================
df_final = df_stage[['cust_id', 'waterfall_hit']].copy()

print(df_final)
df_final.info()

   cust_id waterfall_hit
0        1       rule_10
1        2       rule_10
2        3       rule_10
3        4       rule_18
4        5      rule_999
5        6       rule_15
6        7       rule_10
7        8       rule_10
8        9      rule_999
9       10       rule_18
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cust_id        10 non-null     int64 
 1   waterfall_hit  10 non-null     object
dtypes: int64(1), object(1)
memory usage: 288.0+ bytes


In [12]:
# ==============================================================================
# Step 9: Display the value counts of 'waterfall_hit' from df_final
# ==============================================================================
# Calculate value counts and sort by the rule name.
final_counts = df_final['waterfall_hit'].value_counts().sort_index()

# Convert to DataFrame for better display formatting.
df_final_counts = final_counts.reset_index()
df_final_counts.columns = ['waterfall_hit', 'count']

# Apply thousand comma formatting.
df_final_counts['count'] = df_final_counts['count'].apply(lambda x: f"{x:,}")

print("--- Final Customer Counts by Hit Rule ---")
print(df_final_counts.to_string(index=False))

--- Final Customer Counts by Hit Rule ---
waterfall_hit count
      rule_10     5
      rule_15     1
      rule_18     2
     rule_999     2
